In [1]:
import os
import pandas as pd

In [2]:
os.getcwd()

'C:\\Users\\avari\\Desktop\\drug-likeness-adme-screener'

In [3]:
import os

print("data exists:", os.path.isdir("data"))
print("data/raw exists:", os.path.isdir("data/raw"))
print("data/processed exists:", os.path.isdir("data/processed"))
print("data/results exists:", os.path.isdir("data/results"))
print("notebooks exists:", os.path.isdir("notebooks"))


data exists: False
data/raw exists: False
data/processed exists: False
data/results exists: False
notebooks exists: False


In [4]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/results", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)

print("Folders created.")

Folders created.


In [5]:
print("data exists:", os.path.isdir("data"))
print("data/raw exists:", os.path.isdir("data/raw"))
print("data/processed exists:", os.path.isdir("data/processed"))
print("data/results exists:", os.path.isdir("data/results"))
print("notebooks exists:", os.path.isdir("notebooks"))

data exists: True
data/raw exists: True
data/processed exists: True
data/results exists: True
notebooks exists: True


In [6]:
os.listdir(".")

['.ipynb_checkpoints', 'data', 'notebooks', 'Untitled.ipynb']

In [7]:
import os
os.listdir("data/raw")

['descriptor_table.csv']

In [8]:
import pandas as pd

desc_path = "data/raw/descriptor_table.csv"
df = pd.read_csv(desc_path)

df.head()
df.columns.tolist()

['molecular_weight',
 'alogp',
 'topological_polar_surface_area',
 'hydrogen_bond_donors',
 'hydrogen_bond_acceptors',
 'MolWt_rdkit',
 'LogP_rdkit',
 'TPSA_rdkit',
 'HDonors_rdkit',
 'HAcceptors_rdkit',
 'RotBonds_rdkit']

In [9]:
def lipinski_violations(row):
    violations = 0
    
    if row["MolWt_rdkit"] > 500:
        violations += 1
    if row["LogP_rdkit"] > 5:
        violations += 1
    if row["HDonors_rdkit"] > 5:
        violations += 1
    if row["HAcceptors_rdkit"] > 10:
        violations += 1
        
    return violations

df["Lipinski_violations"] = df.apply(lipinski_violations, axis=1)
df["Lipinski_pass"] = df["Lipinski_violations"] <= 1

df["Veber_pass"] = (
    (df["RotBonds_rdkit"] <= 10) &
    (df["TPSA_rdkit"] <= 140)
)

df[[
    "MolWt_rdkit",
    "LogP_rdkit",
    "TPSA_rdkit",
    "HDonors_rdkit",
    "HAcceptors_rdkit",
    "RotBonds_rdkit",
    "Lipinski_violations",
    "Lipinski_pass",
    "Veber_pass"
]].head()

,MolWt_rdkit,LogP_rdkit,TPSA_rdkit,HDonors_rdkit,HAcceptors_rdkit,RotBonds_rdkit,Lipinski_violations,Lipinski_pass,Veber_pass
0,448.512,1.9746,116.20,1.0,8.0,4.0,0,True,True
1,1079.815,23.3345,78.90,0.0,6.0,63.0,2,False,False
2,340.380,-2.7349,153.78,6.0,6.0,5.0,1,True,False
3,708.808,7.3296,89.55,0.0,11.0,10.0,3,False,True
4,369.417,3.1932,49.39,0.0,6.0,3.0,0,True,True


In [10]:
print("Total compounds:", len(df))
print("Lipinski pass:", df["Lipinski_pass"].sum())
print("Veber pass:", df["Veber_pass"].sum())
print("Pass both:", ((df["Lipinski_pass"]) & (df["Veber_pass"])).sum())

Total compounds: 2000
Lipinski pass: 1314
Veber pass: 1226
Pass both: 1157


In [11]:
# Sort by Lipinski_violations (fewest first), then by MolWt_rdkit just for consistency
df_sorted = df.sort_values(
    by=["Lipinski_violations", "MolWt_rdkit"],
    ascending=[True, True]
)

# Save full screened table
df_sorted.to_csv("data/processed/screened_compounds.csv", index=False)

# Define triage labels
def label_compound(row):
    if row["Lipinski_pass"] and row["Veber_pass"] and row["Lipinski_violations"] == 0:
        return "High-priority"
    elif row["Lipinski_pass"] or row["Veber_pass"]:
        return "Borderline"
    else:
        return "Lower-priority"

df_sorted["Triage_label"] = df_sorted.apply(label_compound, axis=1)

# Save top hits (high-priority only)
top_hits = df_sorted[df_sorted["Triage_label"] == "High-priority"].copy()
top_hits.to_csv("data/processed/top_hits.csv", index=False)

len(top_hits)

921

In [12]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

summary = df_sorted.groupby("Triage_label").size().reset_index(name="Count")

plt.figure(figsize=(7,5))
sns.barplot(data=summary, x="Triage_label", y="Count")
plt.title("Compound Triage Summary")
plt.xlabel("Triage category")
plt.ylabel("Number of compounds")
plt.tight_layout()
plt.savefig("data/results/triage_counts.png", dpi=300)
plt.close()

summary

,Triage_label,Count
0,Borderline,462
1,High-priority,921
2,Lower-priority,617


In [13]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df_sorted,
    x="MolWt_rdkit",
    y="LogP_rdkit",
    hue="Triage_label",
    alpha=0.7
)
plt.title("MW vs LogP Colored by Triage Label")
plt.xlabel("Molecular weight (RDKit)")
plt.ylabel("LogP (RDKit)")
plt.tight_layout()
plt.savefig("data/results/mw_vs_logp_triage.png", dpi=300)
plt.close()

In [14]:
import os
os.listdir("data/results")

['mw_vs_logp_triage.png', 'triage_counts.png']

## Methods

This project screened a 2,000-compound natural-products subset using descriptor-based drug-likeness rules in Python. A precomputed descriptor table was used as input, containing RDKit-derived molecular weight (MolWt_rdkit), LogP (LogP_rdkit), topological polar surface area (TPSA_rdkit), hydrogen bond donors (HDonors_rdkit), hydrogen bond acceptors (HAcceptors_rdkit), and rotatable bonds (RotBonds_rdkit).

A Lipinski-style filter was applied using the following thresholds: molecular weight <= 500, LogP <= 5, hydrogen bond donors <= 5, and hydrogen bond acceptors <= 10. A Veber-style filter was also applied using rotatable bonds <= 10 and TPSA <= 140. Compounds were then labeled as High-priority, Borderline, or Lower-priority based on combined rule performance.

## Results

Out of 2,000 compounds, 1,314 passed the Lipinski-style filter and 1,226 passed the Veber-style filter. A total of 1,157 compounds satisfied both criteria. Using a simple triage scheme, 921 compounds were labeled High-priority, 462 were labeled Borderline, and 617 were labeled Lower-priority.

Two result figures were generated to summarize the screening workflow. A bar chart of triage category counts showed the distribution of compounds across priority levels, and a molecular weight versus LogP scatterplot illustrated how compound classes separated in physicochemical space. These results demonstrate a simple rule-based workflow for prioritizing natural products as oral small-molecule starting points.